In [2]:
import pandas as pd
import json

# 根据eval_results.csv和cluster.json将实例f1score按照簇内元素顺序输出
# out_dir = 'out_ano'
# exp_key = 'use_center_2022_2daytrain'
# exp_key = 'offline_2022_2daytrain'
# exp_key = 'use_center_data2_2021_2daytrain'
# exp_key = 'use_center_modeldim50_2022_2daytrain'
# exp_key = 'use_center_modeldim100_2022_2daytrain'
# exp_key = 'finetune_freeze_rnn_2022_2daytrain'
# exp_key = 'finetune_freeze_rnn_init_random_2022_2daytrain'
# exp_key = 'noshare_2022_7daytrain'
# exp_key = 'finetune_freeze_except_rnn_init_random_2022_2daytrain'
# exp_key = 'finetune_freeze_except_rnn_2022_2daytrain'
exp_key = 'freeze_pnf_validr_epoch10_2021_2daytrain_modeldim500_weight10'
algorithm_name = 'OA_torch'
# bf_results = pd.read_csv(f'/home/chengdaguo/code/Compared_zhujun/{algorithm_name}/{out_dir}/{exp_key}/evaluation_result/bf_machine_best_f1.csv')
bf_results = pd.read_csv(f'/home/chengdaguo/code/Compared_zhujun/IF_pytorch/out_ano/data2/{exp_key}/evaluation_result/bf_machine_best_f1.csv')
cluster_config = json.load(open('/home/chengdaguo/code/Compared_zhujun/exp_data/config/cluster_data2_ano_meanstd_bk.json'))


machine_f1_dict = {}
for index, row in bf_results.iterrows():
    machine_f1_dict[int(row['machine_id'])] = row
with open(f'/home/chengdaguo/code/Compared_zhujun/IF_pytorch/utils_code/{algorithm_name}_{exp_key}.txt', mode='w') as f:
    for cluster_index, cluster in enumerate(cluster_config):
        tp_sum, fp_sum, fn_sum = 0, 0, 0
        for test_id_index, test_id in enumerate(cluster['test']):
            row = machine_f1_dict[test_id]
            tp_sum += row['tp']
            fp_sum += row['fp']
            fn_sum += row['fn']
            
        p = tp_sum / (tp_sum + fp_sum)
        r = tp_sum / (tp_sum + fn_sum)
        f1 = 2*p*r/(p+r)
        print(f"cluster_index: {cluster['label']}---, tp: {tp_sum} fp: {fp_sum} fn: {fn_sum} f1:{round(f1, 4)}", file=f)

        for test_id_index, test_id in enumerate(cluster['test']):
            row = machine_f1_dict[test_id]
            if not (machine_f1_dict[test_id]['f1']==0 and machine_f1_dict[test_id]['fn'] == 0):
                print(f"test: {test_id}\t f1: {row['f1']}\t tp:{row['tp']}\t fp:{row['fp']}\t fn:{row['fn']}\t dis:{cluster['distance'][test_id_index]}", file=f)

/home/chengdaguo/anaconda3/envs/interfusion/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning: invalid value encountered in double_scalars


In [1]:
# 对比多个exp的eval_results


import pandas as pd
import pathlib
from matplotlib import pyplot as plt
import numpy as np

path_list = [
    pathlib.Path('/home/chengdaguo/code/Compared_zhujun/IF_pytorch/out_ocat/yidong/use_center_2021_1daytrain_modeldim500_epoch20/evaluation_result/bf_machine_best_f1.csv'),
    pathlib.Path('/home/chengdaguo/code/Compared_zhujun/IF_pytorch/out_ocat/yidong/freeze_cnn_2021_1daytrain_modeldim500_epoch10/evaluation_result/bf_machine_best_f1.csv'),
    pathlib.Path('/home/chengdaguo/code/Compared_zhujun/IF_pytorch/out_ocat/yidong/freeze_rnn_2021_1daytrain_modeldim500_epoch10/evaluation_result/bf_machine_best_f1.csv'),
    ]
df_list = []
for path in path_list:
    df_list.append(pd.read_csv(path))
col_index_list = [6,1,2,3]


df_merge =  pd.DataFrame()
for index, col in enumerate(col_index_list):
    for df_index, df in enumerate(df_list):
        if index == 0:
            tp = np.sum(df['tp'].values)
            fp = np.sum(df['fp'].values)
            fn = np.sum(df['fn'].values)
            p = tp / (tp+fp)
            r = tp / (tp+fn)
            f1 = 2*p*r/(p+r)
            print(f"{path_list[df_index].parent.parent.name} \t tp:{tp}\t fp:{fp}\t fn:{fn}\t p:{round(p, 4)}\t r:{round(r, 4)}\t f1:{round(f1, 4)}")
        columns = df.columns
        if df_merge.empty:
            df_merge['machine_id'] = df['machine_id']
        # print(df[columns[col]].shape)
        new_col = f"{columns[col]}_{df_index}"
        df_merge[new_col] = df[columns[col]]

df_merge.to_csv("/home/chengdaguo/code/Compared_zhujun/IF_pytorch/utils_code/compare.csv", index=False)

use_center_2021_1daytrain_modeldim500_epoch20 	 tp:1174	 fp:446	 fn:58	 p:0.7247	 r:0.9529	 f1:0.8233
freeze_cnn_2021_1daytrain_modeldim500_epoch10 	 tp:1171	 fp:444	 fn:61	 p:0.7251	 r:0.9505	 f1:0.8226
freeze_rnn_2021_1daytrain_modeldim500_epoch10 	 tp:1179	 fp:432	 fn:53	 p:0.7318	 r:0.957	 f1:0.8294
